# 07. Resampling

Speech models are trained at 16 kHz, CDs are 44.1 kHz, video is 48 kHz: almost every audio
pipeline converts sample rates somewhere. Done carelessly, downsampling folds high frequencies
back into the band as *aliases* and upsampling leaves spectral *images*. This notebook
measures both effects on the library's three quality tiers, compares them with
`scipy.signal.resample_poly`, and rebuilds a windowed-sinc decimator by hand.

**Learning outcomes.** After this notebook you can

- predict where an out-of-band component lands after downsampling ($f_a = |f - k f_{out}|$);
- measure alias and image rejection in dB with a single DFT bin;
- explain why the `fast` tier aliases and the `medium` / `high` tiers do not;
- build a windowed-sinc decimator in NumPy and check it against the library and SciPy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as ss
import audio_samples as aus

rng = np.random.default_rng(0)
F64 = np.dtype("float64")  # the generators take a numpy dtype instance
Q = aus.ResamplingQuality


def amplitude(x, f, fs):
    # Amplitude of the component at f Hz from one DFT bin (exact for whole periods).
    n = np.arange(len(x))
    return 2 * np.abs(np.sum(x * np.exp(-2j * np.pi * f * n / fs))) / len(x)


def level_db(x, f, fs, ref):
    return 20 * np.log10(amplitude(x, f, fs) / ref)

## Concept

**Ratio.** Converting from $f_{in}$ to $f_{out}$ (both in Hz) has the dimensionless ratio

$$r = f_{out} / f_{in}$$

The output can only represent frequencies below its Nyquist frequency $f_{out}/2$.

**Downsampling and aliasing.** Keeping every $q$-th sample ($r = 1/q$) samples the signal
again at $f_{out}$. A component at $f$ Hz above $f_{out}/2$ reappears at

$$f_a = |f - k f_{out}|$$

where $k$ is the integer that brings $f_a$ into $[0, f_{out}/2]$ (notebook 01). Once folded it
is indistinguishable from real content, so a resampler must **low-pass filter before
decimating**, with a cutoff at or below $\min(f_{in}, f_{out})/2$.

**Upsampling and imaging.** Raising the rate by $L$ can be modelled as inserting $L - 1$ zeros
between samples, which leaves copies (*images*) of the spectrum at $k f_{in} \pm f$, followed by
a low-pass *interpolation* filter that removes them.

**Ideal band-limited interpolation.** A band-limited signal is fully described by its samples;
its value at time $t$ (seconds) is

$$x(t) = \sum_n x[n]\, \mathrm{sinc}(f_{in} t - n), \qquad \mathrm{sinc}(u) = \frac{\sin(\pi u)}{\pi u}$$

A practical resampler truncates and windows the sinc and scales it so that its cutoff is
$f_c = \min(f_{in}, f_{out})/2$ (slightly less, to leave room for the transition band). One
formula then does both jobs: interpolation and anti-aliasing.

**Linear interpolation** uses a triangular kernel instead. For integer upsampling by $L = 3$
it equals zero-stuffing followed by the FIR $[1, 2, 3, 2, 1]/3$, whose gain at $f$ (Hz) is

$$G(f) = \frac{3 + 4\cos\omega + 2\cos 2\omega}{9}, \qquad \omega = 2\pi f / f_{out}$$

(the extra $1/3$ undoes the kernel's gain of 3 at DC). $G$ is neither flat in the pass band nor
zero at the images.

**Alias (or image) rejection** is the level of the unwanted component relative to the input
component that produced it:

$$R = 20 \log_{10}(A_{alias} / A_{in})\ \text{dB}$$

where $A$ is a sinusoid's amplitude. 0 dB means no rejection at all.

**The library's tiers** (`aus.ResamplingQuality`):

- `fast`: **no anti-aliasing filter.** Integer down-ratios keep every $q$-th sample; other
  ratios use linear interpolation. *This is the default of `AudioSamples.resample`.*
- `medium`: windowed sinc, 128 taps, Hann² window, cutoff $0.95\times$ the lower Nyquist.
- `high`: windowed sinc, 256 taps, Blackman-Harris² window, cubic interpolation of the kernel
  table, same cutoff.

**Time alignment.** A centred sinc filter delays the signal by half its length. The
whole-buffer `resample` removes that delay in whole output frames,
$\mathrm{round}(L_{sinc}\, r/2 - 1)$, where $L_{sinc}$ is the number of sinc taps, so input
sample $n$ lands at output position $n r$ to within half an output frame.

## With the library

Mix a 1 kHz tone (to keep) with a 10 kHz tone (to remove), both at amplitude 0.5, and convert
48 kHz to 16 kHz with each tier. The 10 kHz tone should fold to
$|10\,000 - 16\,000| = 6\,000$ Hz. We measure each component with one DFT bin over the middle
1600 output samples (whole periods of both 1 kHz and 6 kHz, so there is no leakage).

In [ ]:
fs_in, fs_out = 48_000, 16_000
n = np.arange(12_000)  # 250 ms
x = 0.5 * np.sin(2 * np.pi * 1_000 * n / fs_in) + 0.5 * np.sin(2 * np.pi * 10_000 * n / fs_in)
signal = aus.AudioSamples.new_mono(x, fs_in)

mid = slice(1_200, 2_800)  # middle 100 ms of the 16 kHz output
results = {}
outputs = {}
for name in ("fast", "medium", "high"):
    y = signal.resample(fs_out, getattr(Q, name)).to_numpy()
    assert len(y) == 4_000  # ceil(12000 / 3)
    outputs[name] = y
    results[name] = (level_db(y[mid], 1_000, fs_out, 0.5), level_db(y[mid], 6_000, fs_out, 0.5))
    print(f"{name:6s}: 1 kHz {results[name][0]:+.4f} dB, alias at 6 kHz {results[name][1]:+.1f} dB")

# Every tier passes 1 kHz unchanged (better than 0.001 dB).
assert all(abs(p) < 1e-3 for p, _ in results.values())
# fast: the alias arrives at full level, 0 dB.
assert abs(results["fast"][1]) < 0.01
# medium rejects it by about 102 dB, high by about 142 dB (close to the f32 noise floor:
# the resampler computes in f32). 1 dB tolerances.
assert abs(results["medium"][1] + 101.8) < 1.0
assert abs(results["high"][1] + 141.6) < 1.0

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
win = np.hanning(1_600)
freqs = np.fft.rfftfreq(1_600, 1 / fs_out)
for name, y in outputs.items():
    spec = np.abs(np.fft.rfft(y[mid] * win)) / win.sum() * 2
    ax.plot(freqs, 20 * np.log10(spec / 0.5 + 1e-12), label=name, lw=1)
ax.set(xlabel="Frequency (Hz)", ylabel="Level re input (dB)", ylim=(-170, 10),
       title="48 kHz -> 16 kHz: 1 kHz kept, 10 kHz aliases to 6 kHz")
ax.legend(fontsize=8)
plt.show()

**Compare with SciPy.** `scipy.signal.resample_poly(x, 1, 3)` uses a polyphase FIR designed by
`firwin` with a Kaiser window ($\beta = 5$) and $2 \cdot 10 \cdot \max(\text{up}, \text{down}) + 1 = 61$
taps. It is exactly time-aligned (zero phase), but with its default filter it rejects the alias
by only about 57 dB: a different trade-off, not a bug.

In [ ]:
y_poly = ss.resample_poly(x, 1, 3)
assert len(y_poly) == 4_000
pass_poly, alias_poly = level_db(y_poly[mid], 1_000, fs_out, 0.5), level_db(y_poly[mid], 6_000, fs_out, 0.5)
print(f"resample_poly: 1 kHz {pass_poly:+.4f} dB, alias {alias_poly:+.1f} dB")
assert abs(pass_poly) < 0.01  # pass band intact (0.01 dB)
assert abs(alias_poly + 57.3) < 0.5  # about 57 dB of rejection
# Both sinc tiers beat SciPy's default filter by more than 40 dB here.
assert results["medium"][1] < alias_poly - 40 and results["high"][1] < alias_poly - 40

**Time alignment.** An impulse at input sample 3000 (48 kHz) peaks at output sample
$3000/3 = 1000$ (16 kHz) for both sinc tiers. The alignment is exact only to within half an
output frame: for `high` the removed delay is $\mathrm{round}(256/3/2 - 1) = 42$ frames
against an exact $41.667$, so the output *leads* by one third of an output frame; for `medium`
it is $\mathrm{round}(128/3/2 - 1) = 20$ against $20.333$, so the output *lags* by one third.
We measure that from the phase of a 1 kHz tone ($+0.335$ frames for `high`, $-0.331$ for
`medium`).

In [ ]:
imp = np.zeros(12_000)
imp[3_000] = 1.0
for name in ("medium", "high"):
    y = aus.AudioSamples.new_mono(imp, fs_in).resample(fs_out, getattr(Q, name)).to_numpy()
    assert np.argmax(np.abs(y)) == 1_000

exact_high, exact_medium = 256 / 3 / 2 - 1, 128 / 3 / 2 - 1
assert round(exact_high) == 42 and abs(exact_high - 41.667) < 1e-3
assert round(exact_medium) == 20 and abs(exact_medium - 20.333) < 1e-3
tone = aus.sine_wave(1_000.0, 0.25, sample_rate=fs_in, amplitude=0.5, dtype=F64)
k = np.arange(1_200, 2_800)
shift = {}
for name in ("medium", "high"):
    y = tone.resample(fs_out, getattr(Q, name)).to_numpy()[k]
    phase = np.angle(np.sum(y * np.exp(-2j * np.pi * 1_000 * k / fs_out))) + np.pi / 2
    shift[name] = phase / (2 * np.pi * 1_000 / fs_out)  # in output frames; > 0 means early
print({k_: round(float(v), 3) for k_, v in shift.items()})
# Predicted +-1/3 frame; 0.01 frames covers the filters' own tiny phase error.
assert abs(shift["high"] - 0.335) < 0.01 and abs(shift["medium"] + 0.331) < 0.01

## Under the hood

**`fast` is plain decimation.** With $r = 1/3$ it returns $x[3m]$, and
$\sin(2\pi\, 10\,000\, m / 16\,000) = -\sin(2\pi\, 6\,000\, m / 16\,000)$ for every integer $m$:
the 10 kHz tone *is* a 6 kHz tone on the new grid.

In [ ]:
x10 = aus.sine_wave(10_000.0, 0.1, sample_rate=fs_in, amplitude=0.5, dtype=F64)
y10 = x10.resample(fs_out, Q.fast).to_numpy()
m = np.arange(len(y10))
# The resampler works in f32, hence 1e-6.
assert np.allclose(y10, x10.to_numpy()[::3], atol=1e-6)
assert np.allclose(y10, -0.5 * np.sin(2 * np.pi * 6_000 * m / fs_out), atol=1e-6)

**A windowed-sinc decimator, written out.** Low-pass with a 193-tap Blackman-windowed sinc whose
cutoff is $0.95 \times 8$ kHz $= 7.6$ kHz, then keep every third sample. The filter is centred
(no delay), so the 1 kHz output matches the ideal 16 kHz sine, and the 6 kHz alias is more than
100 dB down, in the same class as the `medium` tier.

In [ ]:
half = 96
fc = 0.95 * (fs_out / 2) / fs_in  # cutoff as a fraction of fs_in
t = np.arange(-half, half + 1)
h = 2 * fc * np.sinc(2 * fc * t) * np.blackman(2 * half + 1)  # np.sinc(u) = sin(pi u)/(pi u)

# y[m] = sum_k h[k] x[3m + half - k]: full convolution, then keep the centred samples.
y_mine = np.convolve(x, h)[half::3][:4_000]

assert abs(level_db(y_mine[mid], 1_000, fs_out, 0.5)) < 1e-3  # pass band intact
assert level_db(y_mine[mid], 6_000, fs_out, 0.5) < -100  # alias > 100 dB down
ideal = 0.5 * np.sin(2 * np.pi * 1_000 * np.arange(4_000) / fs_out)
# Matches the ideal 16 kHz sine to 1e-5 (window ripple in the pass band).
assert np.max(np.abs(y_mine[mid] - ideal[mid])) < 1e-5

fig, ax = plt.subplots(figsize=(8, 3))
H = np.fft.rfft(h, 8192)
ax.plot(np.fft.rfftfreq(8192, 1 / fs_in), 20 * np.log10(np.abs(H) + 1e-12))
ax.axvline(8_000, color="grey", ls=":", lw=0.8)
ax.set(xlim=(0, 24_000), ylim=(-160, 5), xlabel="Frequency (Hz)", ylabel="Gain (dB)",
       title="193-tap Blackman-windowed sinc, cutoff 7.6 kHz (new Nyquist dotted)")
plt.show()

**Linear interpolation is a poor interpolation filter.** Upsampling a 6 kHz tone from 16 kHz
to 48 kHz with `fast` should, by $G(f)$, droop the tone to $G(6\,000) = -3.77$ dB and leave an
image at $16\,000 - 6\,000 = 10\,000$ Hz at $G(10\,000) = -11.8$ dB. The sinc tiers pass the
tone flat and push the image below $-95$ dB ($-98$ dB for `medium`, $-145$ dB for `high`).

In [ ]:
def G_db(f, fs=48_000):
    w = 2 * np.pi * f / fs
    return 20 * np.log10(abs((3 + 4 * np.cos(w) + 2 * np.cos(2 * w)) / 9))

assert abs(G_db(6_000) + 3.77) < 0.01 and abs(G_db(10_000) + 11.8) < 0.05

x6 = aus.sine_wave(6_000.0, 0.25, sample_rate=16_000, amplitude=0.5, dtype=F64)
seg = slice(3_600, 8_400)  # 100 ms: whole periods of 6 and 10 kHz
up = {name: x6.resample(48_000, getattr(Q, name)).to_numpy()[seg] for name in ("fast", "medium", "high")}
lv = {name: (level_db(y, 6_000, 48_000, 0.5), level_db(y, 10_000, 48_000, 0.5)) for name, y in up.items()}
print({name: tuple(round(float(v), 2) for v in vals) for name, vals in lv.items()})
# Measured equals predicted to 0.01 dB (the resampler computes in f32).
assert abs(lv["fast"][0] - G_db(6_000)) < 0.01 and abs(lv["fast"][1] - G_db(10_000)) < 0.01
# The sinc tiers: tone flat to 0.01 dB, image far down.
assert abs(lv["medium"][0]) < 0.01 and abs(lv["high"][0]) < 0.01
assert abs(lv["medium"][1] + 98) < 1.0 and abs(lv["high"][1] + 145) < 1.0

## Ask the library to explain

`aus.explain.resampling(from_rate, to_rate)` summarises a conversion: ratio, both Nyquist
frequencies, the pass-band edge of the sinc tiers, and a warning when you downsample.

In [ ]:
doc = aus.explain.resampling(48_000.0, 16_000.0)
doc

In [ ]:
assert abs(doc.quantity("Ratio") - 1 / 3) < 1e-12
assert doc.quantity("Output Nyquist") == 8_000.0
# 0.95 x 8 kHz; the 0.95 is stored as f32, hence the 1e-3 Hz tolerance.
assert abs(doc.quantity("Sinc pass-band edge") - 7_600.0) < 1e-3
assert doc.has_warning_containing("Downsampling")
# Upsampling discards nothing, so there is no such warning.
assert not aus.explain.resampling(16_000.0, 48_000.0).has_warning_containing("Downsampling")

## Exercises

**1. Where does 13 kHz go?** A 13 kHz tone (amplitude 0.5) is converted from 48 kHz to 16 kHz
with `fast`. Predict the output frequency, confirm it with `aus.explain.aliasing`, and measure
the output amplitude at that frequency. Put the alias frequency in Hz in `answer`.

In [ ]:
answer = None  # TODO

In [ ]:
# Solution
answer = abs(13_000 - 16_000)
assert aus.explain.aliasing(13_000.0, 16_000.0).quantity("Alias frequency") == answer == 3_000
x13 = aus.sine_wave(13_000.0, 0.1, sample_rate=48_000, amplitude=0.5, dtype=F64)
y13 = x13.resample(16_000, Q.fast).to_numpy()  # 1600 samples = 300 periods of 3 kHz
# Full level: fast has no anti-aliasing filter (1e-6: f32 arithmetic).
assert abs(amplitude(y13, 3_000, 16_000) - 0.5) < 1e-6

**2. Droop at 2 kHz.** Predict the gain (dB) of `fast` upsampling 16 kHz to 48 kHz at 2 kHz
from $G(f)$, then measure it. Put the measured gain in `answer`.

In [ ]:
answer = None  # TODO

In [ ]:
# Solution
# w = 2 pi 2000 / 48000 = pi / 12, G = (3 + 4 cos(pi/12) + 2 cos(pi/6)) / 9 = 0.955.
predicted = G_db(2_000)
assert abs(predicted + 0.40) < 0.005
x2 = aus.sine_wave(2_000.0, 0.25, sample_rate=16_000, amplitude=0.5, dtype=F64)
y2 = x2.resample(48_000, Q.fast).to_numpy()[3_600:8_400]
answer = level_db(y2, 2_000, 48_000, 0.5)
assert abs(answer - predicted) < 0.01

**3. When is the alignment exact?** For 48 kHz to 24 kHz with `high` ($L_{sinc} = 256$),
compute $L_{sinc}\, r/2 - 1$. Is the removed delay exact? Measure the largest deviation of a
1 kHz tone (amplitude 0.5) from the ideal 24 kHz sine in the middle 100 ms and put it in
`answer`.

In [ ]:
answer = None  # TODO

In [ ]:
# Solution
exact = 256 * 0.5 / 2 - 1
assert exact == 63 == round(exact)  # an integer: rounding loses nothing
y24 = tone.resample(24_000, Q.high).to_numpy()
kk = np.arange(1_200, 3_600)
answer = np.max(np.abs(y24[kk] - 0.5 * np.sin(2 * np.pi * 1_000 * kk / 24_000)))
# A shift of tau output frames causes an error of up to 0.5 * (2 pi / 24) * tau, so an
# error below 1e-3 means tau < 0.008 frames.
assert answer < 1e-3

## Common pitfalls

- **`fast` is the default** of `AudioSamples.resample` and `resample_by_ratio`, and it has
  **no anti-aliasing filter at all** when downsampling. Pass `aus.ResamplingQuality.medium` or
  `.high` whenever the input may contain energy above the new Nyquist frequency.
- **Resampling is done in `float32`**, so rejection bottoms out around $-140$ dB whatever the
  sample type of your buffer.
- **SciPy and librosa differ in defaults.** `scipy.signal.resample` uses the FFT (assuming a
  periodic signal); `resample_poly` uses a short Kaiser-windowed FIR (about 57 dB of alias
  rejection above); `librosa.resample` uses soxr by default. Expect small differences in edge
  behaviour and alignment, not in the pass band.
- **Alignment is to within half an output frame** (here $\pm 1/3$ frame), not exact, unless
  $L_{sinc} r/2 - 1$ is an integer.
- **Output length** is $\lceil N r \rceil$ frames.

## See also

- Rust guide chapter: <https://docs.rs/audio_samples/latest/audio_samples/guide/c07_resampling/index.html> (available from the next release; until then build the docs locally with `cargo doc --features educational,full_no_plotting --open`)
  and the lab `examples/tutorial_07_resampling.rs`.
- Python API: `AudioSamples.resample`, `AudioSamples.resample_by_ratio`,
  `audio_samples.ResamplingQuality`, `aus.explain.resampling`, `aus.explain.aliasing`.
- Previous: [06. IIR filters](06_iir_filters.ipynb); next:
  [08. LPC, the source-filter model and formants](08_lpc_formants.ipynb); aliasing basics:
  [01. Sampling and aliasing](01_sampling_aliasing.ipynb).

## References

* Crochiere, R. E. and Rabiner, L. R. (1983). *Multirate Digital Signal Processing*. Prentice-Hall.
* Oppenheim, A. V. and Schafer, R. W. (2010). *Discrete-Time Signal Processing*, 3rd ed. Pearson. Section 4.6 (changing the sampling rate by an integer or rational factor).